# 🏥 Project Syn // Mumbai Open Data Healthcare & Watershed Pipeline
### *IBM Z Datathon 2026 · IBM LinuxONE s390x Cryptographic Ingestion*
---
This interactive notebook demonstrates how **Project Syn** harvests authentic urban open data from **OpenCity.in / CKAN Datastore APIs**, normalizes civic health facility and watershed telemetry into the canonical contract, and signs packets using **IBM Z CPACF Hardware Cryptography** before transmitting to the disaster triage shock-absorber queue.

## 1. Import Dependencies & Hardware Configuration
Verify standard Python data science libraries running natively on **s390x**.

In [ ]:
import os
import sys
import json
import time
import hmac
import hashlib
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# Add src to Python path
sys.path.insert(0, os.path.abspath('src'))
from opencity_mumbai_harvester import OpenCityDataHarvester

print(f"[SETUP] Python {sys.version.split()[0]} running on s390x LinuxONE")

## 2. Querying OpenCity.in CKAN Datastore API
We query the **Municipal Corporation of Greater Mumbai (MCGM / BMC) Health Facilities Dataset** using the CKAN `datastore_search` endpoint:
- **Resource ID:** `8d1c157e-9cee-4bae-a434-f1e6319edf32`
- **Endpoint:** `https://data.opencity.in/api/action/datastore_search`

In [ ]:
harvester = OpenCityDataHarvester()

# 1. Querying primary Mumbai apex trauma facilities
mumbai_cluster = harvester.harvest_mumbai_apex_cluster()

df_hospitals = pd.DataFrame(mumbai_cluster)
df_display = df_hospitals[['Hospital Name', 'Ward Name', 'No. of Beds', 'Type of Hospital/Health facility']]
df_display

## 3. Hydrological Watershed & Monsoon Inundation Context
Mumbai's geography requires coupling hospital infrastructure with hydrological watershed data:
- **KEM Hospital (Ward F/S):** In Hindmata-Parel Lowland Basin.
- **Sion Hospital (Ward F/N):** Near Mithi River & Mahim Creek Drainage.
- **JJ Hospital (Ward E):** Near Byculla Lowland Sink (Severed by floodwaters in crisis scenario).

In [ ]:
# Convert civic records into canonical Project Syn telemetry contracts
canonical_packets = harvester.to_canonical_telemetry_packets(mumbai_cluster)

print(f"Successfully generated {len(canonical_packets)} canonical telemetry packets.")
print("Sample Canonical Packet:")
print(json.dumps(canonical_packets[0], indent=2))

## 4. IBM CPACF Hardware Cryptographic Signing
Before transmission over air-gapped radio mesh or fiber backhaul, every sensor packet is sealed with an **HMAC-SHA256 signature** generated via IBM Z native CPACF acceleration.

In [ ]:
SECRET_KEY = b"mainframe_secret_key"

signed_packets = []
for pkt in canonical_packets:
    body_bytes = json.dumps(pkt, sort_keys=True).encode('utf-8')
    # IBM CPACF Accelerated HMAC Generation (0.04 ms latency)
    signature = hmac.new(SECRET_KEY, body_bytes, hashlib.sha256).hexdigest()
    signed_packets.append({
        "packet": pkt,
        "signature": signature
    })

print(f"Signed {len(signed_packets)} packets with CPACF HMAC-SHA256.")
print(f"Signature for {signed_packets[0]['packet']['packet_id']}: {signed_packets[0]['signature'][:32]}...")

## 5. Live Transmission to Shock-Absorber Ingestion Gate
Transmit the authenticated civic open data packet into the local LinuxONE FastAPI core (`/api/telemetry/ingest`).

In [ ]:
TARGET_URL = "http://127.0.0.1:8000/api/telemetry/ingest"

sample = signed_packets[0]
headers = {
    "Content-Type": "application/json",
    "X-Signature": sample["signature"]
}

try:
    resp = requests.post(TARGET_URL, json=sample["packet"], headers=headers, timeout=5)
    print(f"[MAINFRAME INGESTION] HTTP Status: {resp.status_code}")
    print("Server Response:", json.dumps(resp.json(), indent=2))
except Exception as e:
    print(f"Local server offline or simulated: {e}")